# Generate synthetic reservation segments

Writes JSON segment files into the pipeline's landing volume so Auto Loader picks them up.
Covers the scheduled-change test cases (unique 1:1 match, many-to-one ambiguity, >60min no-match)
plus bulk background traffic so the detection output looks realistic.

Parameters `catalog`, `schema`, `volume` are passed by the job.

In [ ]:
dbutils.widgets.text("catalog", "serverless_stable_71zsua_catalog")
dbutils.widgets.text("schema", "scheduled_changes")
dbutils.widgets.text("volume", "segment_landing")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

# Catalog is expected to already exist (managed UC catalog). Only create the
# schema + volume, which we own.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

landing = f"/Volumes/{catalog}/{schema}/{volume}/segments"
dbutils.fs.mkdirs(landing)
print("landing zone:", landing)

In [ ]:
import json, random, datetime as dt

random.seed(42)

AIRPORTS = ["JFK","LAX","ORD","MIA","BOS","SFO","ATL","DEN","DFW","SEA","EWR","IAD","LHR","CDG"]
CARRIERS = ["AA","UA","DL","BA","AF"]
OLD_CODES = ["HX","UN"]
NEW_CODES = ["HK","KK","TK"]

rows = []
_id = [0]
def rec():
    _id[0] += 1
    return f"SEG{_id[0]:06d}"

def add(locator, o, d, code, date, time, carrier):
    rows.append({
        "record_id": rec(), "record_locator": locator,
        "origin": o, "destination": d, "code": code,
        "departure_date": date, "departure_time": time, "carrier": carrier,
    })

# --- Curated cases (mirror the customer's notebook) -----------------------
# Unique match -> DETECTED
add("ABC123","JFK","LAX","HX","2026-08-10","09:30","AA")
add("ABC123","JFK","LAX","HK","2026-08-10","10:15","AA")
# One OLD -> two NEW: ambiguous -> DROPPED
add("DEF456","ORD","MIA","UN","2026-08-11","14:00","UA")
add("DEF456","ORD","MIA","HK","2026-08-11","14:20","UA")
add("DEF456","ORD","MIA","TK","2026-08-11","14:45","UA")
# Two OLD -> one NEW: ambiguous -> DROPPED
add("GHI789","BOS","SFO","HX","2026-08-12","12:00","DL")
add("GHI789","BOS","SFO","UN","2026-08-12","12:30","DL")
add("GHI789","BOS","SFO","HK","2026-08-12","12:15","DL")
# Unique match (UN -> KK, tests the bug-fixed codes) -> DETECTED
add("JKL012","ATL","DEN","UN","2026-08-13","11:00","DL")
add("JKL012","ATL","DEN","KK","2026-08-13","10:30","DL")
# >60 min apart -> NO MATCH
add("MNO345","DFW","SEA","HX","2026-08-14","08:00","AA")
add("MNO345","DFW","SEA","HK","2026-08-14","09:30","AA")
print(f"curated rows: {len(rows)}")

In [ ]:
# --- Bulk background traffic ------------------------------------------------
# ~600 PNRs. ~40% are genuine scheduled changes (unique OLD->NEW within 60min),
# the rest are noise: lone confirmations, unrelated segments, and a few
# ambiguous multi-matches. Gives the gold table a realistic hit rate.
base = dt.date(2026, 8, 1)
for i in range(600):
    loc = f"PNR{i:05d}"
    o, d = random.sample(AIRPORTS, 2)
    carrier = random.choice(CARRIERS)
    date = (base + dt.timedelta(days=random.randint(0, 30))).isoformat()
    h, m = random.randint(6, 21), random.choice([0, 15, 30, 45])
    r = random.random()
    if r < 0.40:
        # genuine scheduled change: OLD then NEW within 60 min
        add(loc, o, d, random.choice(OLD_CODES), date, f"{h:02d}:{m:02d}", carrier)
        delta = random.randint(5, 55)
        nt = (dt.datetime(2000,1,1,h,m) + dt.timedelta(minutes=delta)).time()
        add(loc, o, d, random.choice(NEW_CODES), date, nt.strftime("%H:%M"), carrier)
    elif r < 0.70:
        # lone confirmation, no prior cancel -> no match
        add(loc, o, d, random.choice(NEW_CODES), date, f"{h:02d}:{m:02d}", carrier)
    elif r < 0.85:
        # cancel with a NEW too far apart (>60 min) -> no match
        add(loc, o, d, random.choice(OLD_CODES), date, f"{h:02d}:{m:02d}", carrier)
        add(loc, o, d, random.choice(NEW_CODES), date, f"{(h+2)%24:02d}:{m:02d}", carrier)
    else:
        # ambiguous: one OLD, two NEW close together -> dropped
        add(loc, o, d, random.choice(OLD_CODES), date, f"{h:02d}:{m:02d}", carrier)
        add(loc, o, d, random.choice(NEW_CODES), date, f"{h:02d}:{(m+10)%60:02d}", carrier)
        add(loc, o, d, random.choice(NEW_CODES), date, f"{h:02d}:{(m+20)%60:02d}", carrier)

print(f"total rows: {len(rows)}")

In [ ]:
# Write as newline-delimited JSON into a single batch file. A fresh filename
# each run lets Auto Loader treat it as new data on the next pipeline refresh.
ts = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
target = f"{landing}/batch_{ts}.json"
payload = "\n".join(json.dumps(r) for r in rows)
dbutils.fs.put(target, payload, overwrite=True)
print(f"wrote {len(rows)} segments to {target}")